# Notebook 03 — Neural Network Surrogate for Beam Physics

**CAS 2026 · AI for Medical Accelerators**

In this notebook you will:
1. Use **Cheetah** to generate a training dataset (random quadrupole settings → beam properties)
2. Train an **MLP surrogate** that predicts beam properties in < 1 ms (vs. 2 ms per Cheetah call)
3. Measure the speedup and evaluate accuracy (R², MAE)
4. Use the surrogate **inside a Bayesian optimisation loop** — replacing Cheetah entirely
5. Bonus: add uncertainty quantification with a simple ensemble

**Key idea:** Tracking codes can take minutes to hours per run. A surrogate trained on thousands of simulation runs replaces them in milliseconds — enabling real-time control and rapid optimisation.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aghribi/cas2026-ai-medical-accelerators/blob/main/notebooks/03_surrogate_model/notebook.ipynb)

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install -q cheetah-accelerator scikit-optimize plotly

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import cheetah
from skopt import gp_minimize
from skopt.space import Real
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import r2_score

torch.manual_seed(42)
np.random.seed(42)

PROTON_REST_MASS_MEV = 938.272
TOTAL_ENERGY_EV = (150.0 + PROTON_REST_MASS_MEV) * 1e6

print('Imports OK')

## 1 · Rebuild the Cheetah lattice (same as notebook 02)

In [ ]:
segment = cheetah.Segment(elements=[
    cheetah.Drift(length=torch.tensor(0.5)),
    cheetah.Quadrupole(length=torch.tensor(0.1), name="QF"),
    cheetah.Drift(length=torch.tensor(1.0)),
    cheetah.Quadrupole(length=torch.tensor(0.1), name="QD"),
    cheetah.Drift(length=torch.tensor(0.5)),
    cheetah.Screen(name="monitor"),
])

beam_in = cheetah.ParameterBeam.from_twiss(
    energy=torch.tensor(TOTAL_ENERGY_EV),
    beta_x=torch.tensor(1.0),
    beta_y=torch.tensor(1.0),
    emittance_x=torch.tensor(1e-6),
    emittance_y=torch.tensor(1e-6),
    species=cheetah.Species("proton"),
)

print('Lattice and beam ready.')

## 2 · Generate training data with Cheetah

We sweep over random quadrupole strengths (k₁_QF, k₁_QD) and record the beam properties at the monitor.
This is the **slow step** — Cheetah takes ~2 ms per call. 5000 samples ≈ 10 seconds.

In [ ]:
N_TRAIN = 5_000
N_TEST  = 1_000
K1_MIN, K1_MAX = -20.0, 20.0

def run_cheetah(k1_qf: float, k1_qd: float) -> np.ndarray:
    """Single Cheetah evaluation → [σ_x, σ_y, μ_x, μ_y] in mm."""
    segment.QF.k1 = torch.tensor(k1_qf, dtype=torch.float64)
    segment.QD.k1 = torch.tensor(k1_qd, dtype=torch.float64)
    b = segment.track(beam_in)
    return np.array([
        b.sigma_x.item() * 1e3,   # mm
        b.sigma_y.item() * 1e3,
        b.mu_x.item()   * 1e3,
        b.mu_y.item()   * 1e3,
    ])

# Latin Hypercube-like sampling (uniform random for simplicity)
rng = np.random.default_rng(42)
k1_samples_all = rng.uniform(K1_MIN, K1_MAX, size=(N_TRAIN + N_TEST, 2)).astype(np.float32)

print(f'Generating {N_TRAIN + N_TEST} Cheetah evaluations...')
t0 = time.perf_counter()
y_all = np.array([run_cheetah(*params) for params in k1_samples_all], dtype=np.float32)
t_gen = time.perf_counter() - t0
print(f'Done in {t_gen:.1f} s  ({t_gen/(N_TRAIN+N_TEST)*1000:.2f} ms/call)')

# Train / test split
X_train = torch.tensor(k1_samples_all[:N_TRAIN])
y_train = torch.tensor(y_all[:N_TRAIN])
X_test  = torch.tensor(k1_samples_all[N_TRAIN:])
y_test  = torch.tensor(y_all[N_TRAIN:])

print(f'X_train: {X_train.shape}  →  y_train: {y_train.shape}')

### Explore the training data

In [ ]:
# Plot σ_x as a function of (k1_QF, k1_QD)
fig = go.Figure(go.Scatter(
    x=k1_samples_all[:500, 0], y=k1_samples_all[:500, 1],
    mode='markers',
    marker=dict(
        color=y_all[:500, 0],   # σ_x in mm
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='σ_x (mm)'),
        size=5, opacity=0.8,
    ),
))
fig.update_layout(
    title='Beam size σ_x as a function of quadrupole strengths (500 samples)',
    xaxis_title='k₁_QF (T/m²)', yaxis_title='k₁_QD (T/m²)',
    paper_bgcolor='#0d1b2a', plot_bgcolor='#111f30',
    font=dict(color='#e2e8f0'),
)
fig.show()

## 3 · Train the MLP surrogate

A 3-layer network with **Tanh activations** (smoother than ReLU for physics regression).
Input: (k₁_QF, k₁_QD) · Output: (σ_x, σ_y, μ_x, μ_y) in mm.

In [ ]:
class BeamSurrogate(nn.Module):
    """MLP surrogate: (k1_QF, k1_QD) → (σ_x, σ_y, μ_x, μ_y) in mm."""

    def __init__(self, n_in: int = 2, n_out: int = 4, width: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, width), nn.Tanh(),
            nn.Linear(width, width), nn.Tanh(),
            nn.Linear(width, width), nn.Tanh(),
            nn.Linear(width, n_out),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


surrogate = BeamSurrogate()
n_params  = sum(p.numel() for p in surrogate.parameters())
print(f'Surrogate parameters: {n_params:,}')

In [ ]:
# Normalise inputs and outputs for stable training
X_mean, X_std = X_train.mean(0), X_train.std(0)
y_mean, y_std = y_train.mean(0), y_train.std(0)

def normalise_X(x): return (x - X_mean) / X_std
def normalise_y(y): return (y - y_mean) / y_std
def denormalise_y(y): return y * y_std + y_mean

X_train_n = normalise_X(X_train)
y_train_n = normalise_y(y_train)
X_test_n  = normalise_X(X_test)

In [ ]:
EPOCHS = 300
BATCH  = 256

loader    = DataLoader(TensorDataset(X_train_n, y_train_n), batch_size=BATCH, shuffle=True)
optimizer = torch.optim.Adam(surrogate.parameters(), lr=3e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

train_losses, val_losses = [], []
for epoch in range(EPOCHS):
    surrogate.train()
    epoch_loss = 0.0
    for X_b, y_b in loader:
        pred = surrogate(X_b)
        loss = nn.functional.mse_loss(pred, y_b)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item() * len(X_b)
    train_losses.append(epoch_loss / N_TRAIN)
    scheduler.step()

    surrogate.eval()
    with torch.no_grad():
        val_pred = surrogate(X_test_n)
        val_loss = nn.functional.mse_loss(val_pred, normalise_y(y_test)).item()
    val_losses.append(val_loss)

    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS}  train={train_losses[-1]:.6f}  val={val_loss:.6f}')

print('Training complete.')

## 4 · Evaluate accuracy

In [ ]:
surrogate.eval()
with torch.no_grad():
    y_pred = denormalise_y(surrogate(X_test_n)).numpy()
y_true = y_test.numpy()

OUTPUT_NAMES = ['σ_x', 'σ_y', 'μ_x', 'μ_y']
print('Accuracy on held-out test set:')
print(f'{"Output":<6}  {"R²":>6}  {"MAE (mm)":>10}')
print('-' * 28)
for i, name in enumerate(OUTPUT_NAMES):
    r2  = r2_score(y_true[:, i], y_pred[:, i])
    mae = np.abs(y_true[:, i] - y_pred[:, i]).mean()
    print(f'{name:<6}  {r2:>6.4f}  {mae:>10.4f}')

In [ ]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['σ_x: predicted vs. Cheetah', 'σ_y: predicted vs. Cheetah'])

for col, (i, name) in enumerate([(0, 'σ_x'), (1, 'σ_y')], 1):
    lo = min(y_true[:, i].min(), y_pred[:, i].min())
    hi = max(y_true[:, i].max(), y_pred[:, i].max())
    fig.add_trace(go.Scatter(
        x=y_true[:, i], y=y_pred[:, i], mode='markers',
        marker=dict(color='#22d3ee', size=3, opacity=0.5),
        name=name, showlegend=False,
    ), row=1, col=col)
    fig.add_trace(go.Scatter(
        x=[lo, hi], y=[lo, hi], mode='lines',
        line=dict(color='#f87171', dash='dash', width=1),
        name='perfect', showlegend=False,
    ), row=1, col=col)

fig.update_xaxes(title_text='Cheetah (mm)', showgrid=True, gridcolor='#1e3048')
fig.update_yaxes(title_text='Surrogate (mm)', showgrid=True, gridcolor='#1e3048')
fig.update_layout(
    height=420,
    paper_bgcolor='#0d1b2a', plot_bgcolor='#111f30',
    font=dict(color='#e2e8f0'),
)
fig.show()

## 5 · Speed comparison

In [ ]:
N_BENCH = 1_000
bench_params = torch.FloatTensor(N_BENCH, 2).uniform_(K1_MIN, K1_MAX)

# Cheetah: serial calls (one at a time, as in a real control loop)
t0 = time.perf_counter()
for k1_qf, k1_qd in bench_params:
    segment.QF.k1 = k1_qf
    segment.QD.k1 = k1_qd
    _ = segment.track(beam_in)
t_cheetah = time.perf_counter() - t0

# Surrogate: batched inference
bench_n = normalise_X(bench_params)
t0 = time.perf_counter()
surrogate.eval()
with torch.no_grad():
    _ = denormalise_y(surrogate(bench_n))
t_surrogate = time.perf_counter() - t0

print(f'Cheetah   : {t_cheetah*1e3:7.1f} ms for {N_BENCH} evals  ({t_cheetah/N_BENCH*1e3:.3f} ms/call)')
print(f'Surrogate : {t_surrogate*1e3:7.1f} ms for {N_BENCH} evals  ({t_surrogate/N_BENCH*1e6:.1f} μs/call)')
print(f'Speedup   : {t_cheetah/t_surrogate:.0f}×')

# Bar chart
fig = go.Figure(go.Bar(
    x=['Cheetah (serial)', 'Surrogate (batched)'],
    y=[t_cheetah * 1e3, t_surrogate * 1e3],
    marker_color=['#f87171', '#22d3ee'],
    text=[f'{t_cheetah*1e3:.0f} ms', f'{t_surrogate*1e3:.1f} ms'],
    textposition='outside',
))
fig.update_layout(
    title=f'Wall time for {N_BENCH} evaluations',
    yaxis_title='Time (ms)',
    paper_bgcolor='#0d1b2a', plot_bgcolor='#111f30',
    font=dict(color='#e2e8f0'),
)
fig.show()

## 6 · Surrogate-in-the-loop: BO with the surrogate instead of Cheetah

Now we replace Cheetah with our trained surrogate inside the Bayesian optimisation objective.
The surrogate is already 100s× faster — but more importantly, for production use, this scales to expensive tracking codes (TraceWin, G4beamline) where each call takes minutes.

In [ ]:
surrogate.eval()

def objective_surrogate(params: list[float]) -> float:
    """Use the surrogate instead of Cheetah."""
    x = torch.tensor(params, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        y_pred = denormalise_y(surrogate(normalise_X(x)))
    sigma_x, sigma_y = y_pred[0, 0].item(), y_pred[0, 1].item()
    return (sigma_x + sigma_y) * 1e-3   # back to metres for consistency

space = [Real(K1_MIN, K1_MAX, name="k1_QF"), Real(K1_MIN, K1_MAX, name="k1_QD")]

t0 = time.perf_counter()
bo_surrogate = gp_minimize(
    func=objective_surrogate,
    dimensions=space,
    n_calls=40, n_initial_points=8, random_state=42,
)
t_bo_surrogate = time.perf_counter() - t0

# Validate: run the BO result through Cheetah (ground truth)
sigma_ground_truth = sum(run_cheetah(*bo_surrogate.x)[:2])

print(f'BO (surrogate): best σ = {bo_surrogate.fun*1e3:.2f} mm  (wall time: {t_bo_surrogate:.2f} s)')
print(f'Ground truth  : σ = {sigma_ground_truth:.2f} mm  (Cheetah validation)')
print(f'  k1_QF = {bo_surrogate.x[0]:.2f},  k1_QD = {bo_surrogate.x[1]:.2f} T/m²')

---
## Bonus · Ensemble uncertainty quantification

A single surrogate gives predictions but no uncertainty estimate.
An **ensemble of 5 surrogates** trained with different random seeds gives a cheap uncertainty estimate:
high variance = the model is extrapolating, be cautious.

In [ ]:
N_ENSEMBLE = 5
ensemble = []

for seed in range(N_ENSEMBLE):
    torch.manual_seed(seed)
    m = BeamSurrogate()
    opt = torch.optim.Adam(m.parameters(), lr=3e-3)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=200)
    dl  = DataLoader(TensorDataset(X_train_n, y_train_n), batch_size=256, shuffle=True)
    m.train()
    for _ in range(200):
        for X_b, y_b in dl:
            loss = nn.functional.mse_loss(m(X_b), y_b)
            opt.zero_grad(); loss.backward(); opt.step()
        sch.step()
    m.eval()
    ensemble.append(m)
    print(f'  Model {seed+1}/{N_ENSEMBLE} trained')

# Predict with ensemble
with torch.no_grad():
    preds = torch.stack([denormalise_y(m(X_test_n)) for m in ensemble], dim=0)  # (5, N, 4)

mean_pred = preds.mean(0)
std_pred  = preds.std(0)

# Coverage: fraction of true values within ±2σ ensemble
coverage = ((y_test - mean_pred).abs() < 2 * std_pred).float().mean(0)
print('\nEnsemble coverage at ±2σ:')
for i, name in enumerate(OUTPUT_NAMES):
    print(f'  {name}: {coverage[i].item():.2%}')

---
## Summary

| | Cheetah | Surrogate | Speedup |
|---|---|---|---|
| Per-call time | ~2 ms | ~0.01 ms | ~200× |
| Differentiable | ✓ | ✓ | — |
| Works offline | ✓ | ✓ (after training) | — |
| Works on real machine | ✗ | ✓ (if trained on real data) | — |
| Uncertainty | ✗ | ✓ (ensemble) | — |

The surrogate trained on Cheetah is useful for prototyping. In production:
- Train on **real machine data** (or on more expensive codes like TraceWin / Geant4)
- Update the surrogate online as the machine drifts
- Use the ensemble std to flag when you're extrapolating outside the training domain

**Further reading:**
- Kaiser et al., PRAB 27, 054601 (2024) — Cheetah
- Edelen et al., NeurIPS ML4PS (2020) — surrogates in accelerator control
- AccML living review: `aghribi.github.io/acc-ml-living-review`